# Math Photo Solver — Обучение в Google Colab

Этот ноутбук выполняет полный цикл:
1. Клонирует репозиторий
2. Устанавливает зависимости
3. Генерирует датасет символов (80% train / 20% val)
4. Обучает классификатор ResNet-18
5. Оценивает точность модели
6. Сохраняет веса на Google Drive
7. **Генерирует тестовое фото задачи и прогоняет через нейросеть**
8. **Запускает веб-сайт с публичным URL для ручного тестирования**

> **Перед запуском**: `Среда выполнения → Сменить тип среды выполнения → GPU (T4) → Сохранить`

## Шаг 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/math_solver_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Google Drive подключён. Модель будет сохранена в:', DRIVE_SAVE_DIR)

## Шаг 2. Клонирование репозитория

In [ ]:
# Замените URL на адрес своего репозитория
REPO_URL = 'https://github.com/sergey2321/sergey2321.git'

!git clone {REPO_URL} /content/math-solver
%cd /content/math-solver
!git checkout claude/ale-yP87K
print('Репозиторий склонирован.')

## Шаг 3. Установка зависимостей

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok  # для публичного URL сайта
print('Зависимости установлены.')

## Шаг 4. Проверка GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ВНИМАНИЕ: GPU не найден. Обучение будет медленным.')

## Шаг 5. Генерация датасета символов (80/20)

In [ ]:
# Количество изображений — увеличь для лучшей точности (рекомендуется 20000+)
DATASET_COUNT = 10000
DATASET_DIR   = 'dataset/symbols'

!python -m dataset.generator.generate_handwritten \
    --count {DATASET_COUNT} \
    --out {DATASET_DIR} \
    --seed 42

import os
train_count = len(os.listdir(f'{DATASET_DIR}/train'))
val_count   = len(os.listdir(f'{DATASET_DIR}/val'))
print(f'Train: {train_count}  |  Val: {val_count}')

## Шаг 6. Обучение модели

In [ ]:
EPOCHS     = 25
BATCH_SIZE = 128
LR         = 1e-3
MODEL_OUT  = 'backend/models/symbol_clf.pth'

!python -m training.train_ocr \
    --data {DATASET_DIR} \
    --epochs {EPOCHS} \
    --batch {BATCH_SIZE} \
    --lr {LR} \
    --out {MODEL_OUT}

## Шаг 7. Оценка точности модели

In [ ]:
!python -m training.evaluate \
    --data {DATASET_DIR} \
    --model {MODEL_OUT}

## Шаг 8. Сохранение модели на Google Drive

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{DRIVE_SAVE_DIR}/symbol_clf_{timestamp}.pth'
shutil.copy(MODEL_OUT, dest)
print(f'Модель сохранена на Drive: {dest}')

shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_latest.pth')
print(f'Также сохранена как: {DRIVE_SAVE_DIR}/symbol_clf_latest.pth')

## Шаг 9. Скачивание модели на локальный компьютер

После завершения обучения скачай веса одним из способов:

**Способ 1** — Скачать прямо из Colab (раскомментируй ячейку ниже).

**Способ 2** — Взять с Google Drive:  
`Мой диск → math_solver_models → symbol_clf_latest.pth`

Положи скачанный файл в `backend/models/symbol_clf.pth` в локальном репозитории.

In [ ]:
# Раскомментируй для прямого скачивания
# from google.colab import files
# files.download('backend/models/symbol_clf.pth')

---
## Шаг 10. Генерация тестового фото и проверка нейросети

Генерируем картинку с математической задачей прямо в Colab и прогоняем через нейросеть.
Никакой загрузки файлов не нужно — всё автоматически.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import os

# Список тестовых задач — можешь добавить свои
TEST_EXPRESSIONS = [
    '2 + 2',
    '15 * 3 - 7',
    '2*x + 3 = 7',
    'x**2 - 5*x + 6 = 0',
    'integrate(x**2, x)',
    'diff(x**3 + 2*x, x)',
]

os.makedirs('test_images', exist_ok=True)

def render_test_image(expression, path, width=400, height=80, font_size=32):
    img = Image.new('L', (width, height), color=255)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuMono.ttf', font_size)
    except:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), expression, font=font)
    x = max(4, (width  - (bbox[2]-bbox[0])) // 2)
    y = max(4, (height - (bbox[3]-bbox[1])) // 2)
    draw.text((x, y), expression, fill=0, font=font)
    img.save(path)
    return img

# Генерируем и показываем все тестовые картинки
fig, axes = plt.subplots(len(TEST_EXPRESSIONS), 1, figsize=(10, len(TEST_EXPRESSIONS)*1.4))
test_paths = []
for i, expr in enumerate(TEST_EXPRESSIONS):
    path = f'test_images/test_{i}.png'
    img = render_test_image(expr, path)
    test_paths.append(path)
    axes[i].imshow(img, cmap='gray', aspect='auto')
    axes[i].set_title(f'Задача {i+1}: {expr}', fontsize=10, loc='left')
    axes[i].axis('off')

plt.suptitle('Тестовые изображения (будут отправлены в нейросеть)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print(f'\nСгенерировано {len(test_paths)} тестовых изображений.')

In [ ]:
# Прогоняем каждую картинку через нейросеть напрямую (без HTTP)
import sys
sys.path.insert(0, '/content/math-solver')

from backend.preprocessing.image_prep import preprocess_image
from backend.ocr.printed import extract_expression
from backend.solver.solver import solve

print('=' * 60)
print('РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ')
print('=' * 60)

for i, path in enumerate(test_paths):
    with open(path, 'rb') as f:
        img_bytes = f.read()

    # 1. Предобработка
    preprocessed = preprocess_image(img_bytes)

    # 2. OCR — распознавание текста
    recognized = extract_expression(preprocessed)

    # 3. Решение
    result = solve(recognized)

    print(f'\nЗадача {i+1}: {TEST_EXPRESSIONS[i]}')
    print(f'  Распознано : {recognized}')
    print(f'  Ответ      : {result["answer"]}')
    print(f'  Проверка   : {"✓ пройдена" if result["verified"] else "✗ не пройдена"}')
    if result['error']:
        print(f'  Ошибка     : {result["error"]}')
    print(f'  Шаги       :')
    for step in result['steps']:
        print(f'    {step}')

print('\n' + '=' * 60)

---
## Шаг 11. Запуск веб-сайта с публичным URL (через ngrok)

Запускает полноценный сайт с интерфейсом прямо в Colab.
Ты получишь публичную ссылку — открой её на телефоне или компьютере,
загрузи любое фото задачи и получи решение.

> **Нужен токен ngrok** — бесплатно на [ngrok.com](https://ngrok.com) → `Your Authtoken`

In [ ]:
# Вставь свой токен ngrok (бесплатно: https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = 'ВСТАВЬ_ТОКЕН_СЮДА'

from pyngrok import ngrok, conf
import subprocess, time

# Установка токена
conf.get_default().auth_token = NGROK_TOKEN

# Запуск FastAPI сервера в фоне
server = subprocess.Popen(
    ['uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/math-solver',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)  # ждём старта

# Создаём публичный тоннель
public_url = ngrok.connect(8000)

print('=' * 55)
print('  Сайт запущен!')
print(f'  Открой в браузере: {public_url}')
print('=' * 55)
print('Загружай фото задачи — получай решение.')
print('Чтобы остановить сервер — выполни следующую ячейку.')

In [ ]:
# Остановка сервера и тоннеля
ngrok.disconnect(public_url)
server.terminate()
print('Сервер остановлен.')

---
## Бонус: Визуализация датасета

Покажем несколько примеров сгенерированных символов.

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
import random

with open(f'{DATASET_DIR}/metadata.json') as f:
    meta = json.load(f)

samples = random.sample(meta['train'], min(20, len(meta['train'])))

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for ax, rec in zip(axes.flat, samples):
    img = Image.open(f"{DATASET_DIR}/{rec['file']}")
    ax.imshow(img, cmap='gray')
    ax.set_title(rec['symbol'], fontsize=10)
    ax.axis('off')

plt.suptitle('Примеры символов из датасета', fontsize=14)
plt.tight_layout()
plt.show()